# MongoDB Marketing Campaign Analysis

## Install pymongo

In [1]:
!pip install pymongo

## Import Libraries

In [2]:
# Import libraries
from pymongo import MongoClient
import pandas as pd

## Connect to MongoDB

In [8]:
# Connect to MongoDB
client = MongoClient('mongodb://localhost:27017/')

# Select the database
db = client["marketing"]

# Select the collection
collection = db["marketing_campaign"]

print("Connected successfully!")
print("Database:", db.name)
print("Collection:", collection.name)

Connected successfully!
Database: marketing
Collection: marketing_campaign


## Count the Total Number of Records

In [9]:
total_records = collection.count_documents({})
print("Total number of records:", total_records)

Total number of records: 2240


## Retrieve the First 3 Records

In [10]:
results = collection.find().limit(3)

for result in results:
    print(result)
    print()

{'_id': ObjectId('6a2623073cdc8e043f452ef2'), 'ID': 5524, 'Year_Birth': 1957, 'Education': 'Graduation', 'Marital_Status': 'Single', 'Income': 58138, 'Kidhome': 0, 'Teenhome': 0, 'Dt_Customer': '2012-09-04', 'Recency': 58, 'MntWines': 635, 'MntFruits': 88, 'MntMeatProducts': 546, 'MntFishProducts': 172, 'MntSweetProducts': 88, 'MntGoldProds': 88, 'NumDealsPurchases': 3, 'NumWebPurchases': 8, 'NumCatalogPurchases': 10, 'NumStorePurchases': 4, 'NumWebVisitsMonth': 7, 'AcceptedCmp3': 0, 'AcceptedCmp4': 0, 'AcceptedCmp5': 0, 'AcceptedCmp1': 0, 'AcceptedCmp2': 0, 'Complain': 0, 'Z_CostContact': 3, 'Z_Revenue': 11, 'Response': 1}

{'_id': ObjectId('6a2623073cdc8e043f452ef3'), 'ID': 2174, 'Year_Birth': 1954, 'Education': 'Graduation', 'Marital_Status': 'Single', 'Income': 46344, 'Kidhome': 1, 'Teenhome': 1, 'Dt_Customer': '2014-03-08', 'Recency': 38, 'MntWines': 11, 'MntFruits': 1, 'MntMeatProducts': 6, 'MntFishProducts': 2, 'MntSweetProducts': 1, 'MntGoldProds': 6, 'NumDealsPurchases': 2, 'N

## Query 1: Find Customers with Income > $70,000
Only print `ID`, `Income`, and `Education`

In [11]:
query = {'Income': {'$gt': 70000}}

results = collection.find(query, {'ID': 1, 'Income': 1, 'Education': 1, '_id': 0}).limit(5)
count = collection.count_documents(query)

for r in results:
    print(f"ID: {r['ID']}, Income: ${r['Income']:,.0f}, Education: {r['Education']}")

print(f"\nTotal customers with Income > $70,000: {count}")

ID: 4141, Income: $71,613, Education: Graduation
ID: 2114, Income: $82,800, Education: PhD
ID: 6565, Income: $76,995, Education: Master
ID: 1966, Income: $84,618, Education: PhD
ID: 8601, Income: $80,011, Education: Graduation

Total customers with Income > $70,000: 508


## Query 2: Customers with Kids AND Teenagers at Home
Combine two conditions using a single query

In [12]:
query = {
    'Kidhome': {'$gte': 1},
    'Teenhome': {'$gte': 1}
}

count = collection.count_documents(query)
results = collection.find(query, {'ID': 1, 'Kidhome': 1, 'Teenhome': 1, 'Marital_Status': 1, '_id': 0}).limit(5)

print(f"Customers with both kids and teens at home: {count}\n")
for r in results:
    print(f"ID: {r['ID']}, Kids: {r['Kidhome']}, Teens: {r['Teenhome']}, Status: {r['Marital_Status']}")

Customers with both kids and teens at home: 427

ID: 2174, Kids: 1, Teens: 1, Status: Single
ID: 5899, Kids: 1, Teens: 1, Status: Together
ID: 8180, Kids: 1, Teens: 1, Status: Divorced
ID: 9736, Kids: 1, Teens: 1, Status: Married
ID: 2404, Kids: 1, Teens: 1, Status: Married


## Query 3: Top Wine Spenders (MntWines > $500)
Use `$gt` operator — only print `ID` and `MntWines`

In [13]:
query = {'MntWines': {'$gt': 500}}

count = collection.count_documents(query)
results = collection.find(query, {'ID': 1, 'MntWines': 1, '_id': 0}).limit(5)

print(f"Customers spending over $500 on wine: {count}\n")
for r in results:
    print(f"ID: {r['ID']}, Wine Spending: ${r['MntWines']}") 

Customers spending over $500 on wine: 565

ID: 5524, Wine Spending: $635
ID: 7446, Wine Spending: $520
ID: 2114, Wine Spending: $1006
ID: 6565, Wine Spending: $1012
ID: 1993, Wine Spending: $867


## Query 4: Recent Customers Who Accepted the Campaign
Find customers with `Recency < 30` days AND `Response = 1`

In [14]:
query = {
    'Recency': {'$lt': 30},
    'Response': 1
}

count = collection.count_documents(query)
results = collection.find(query, {'ID': 1, 'Recency': 1, 'Response': 1, 'Income': 1, '_id': 0}).limit(5)

print(f"Recent customers who responded positively: {count}\n")
for r in results:
    print(f"ID: {r['ID']}, Last Purchase: {r['Recency']} days ago, Income: ${r.get('Income', 0):,.0f}")

Recent customers who responded positively: 166

ID: 4855, Last Purchase: 19 days ago, Income: $30,351
ID: 2114, Last Purchase: 23 days ago, Income: $82,800
ID: 7373, Last Purchase: 8 days ago, Income: $46,610
ID: 9909, Last Purchase: 24 days ago, Income: $7,500
ID: 6853, Last Purchase: 12 days ago, Income: $75,777


## Query 5: Customers Who Filed a Complaint
Find all customers where `Complain = 1`

In [15]:
query = {'Complain': 1}

count = collection.count_documents(query)
results = collection.find(query, {'ID': 1, 'Complain': 1, 'Education': 1, 'Marital_Status': 1, '_id': 0}).limit(5)

print(f"Total customers who complained: {count}\n")
for r in results:
    print(f"ID: {r['ID']}, Education: {r['Education']}, Status: {r['Marital_Status']}")

Total customers who complained: 21

ID: 10401, Education: 2n Cycle, Status: Together
ID: 3120, Education: Graduation, Status: Together
ID: 7829, Education: 2n Cycle, Status: Divorced
ID: 5726, Education: Master, Status: Single
ID: 6201, Education: Graduation, Status: Single


## Aggregation Pipeline: Max & Min Income
Use `$group` with `$max` and `$min` operators

**Pipeline rules:**
1. Every field name goes in quotes `" "`
2. Use `aggregate()` — not `find()`
3. Wrap results in `list()`
4. `"_id"` can be `None` or `""`


In [16]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "max_income": {"$max": "$Income"},
            "min_income": {"$min": "$Income"},
            "avg_income": {"$avg": "$Income"}
        }
    }
]

result = list(collection.aggregate(pipeline))

if result:
    print(f"Max Income: ${result[0]['max_income']:,.2f}")
    print(f"Min Income: ${result[0]['min_income']:,.2f}")
    print(f"Avg Income: ${result[0]['avg_income']:,.2f}")
else:
    print("No data found")

Max Income: $666,666.00
Min Income: $1,730.00
Avg Income: $52,247.25


## Aggregation Pipeline: Total Spending Stats per Education Level

In [18]:
pipeline = [
    {
        "$group": {
            "_id": "$Education",
            "avg_wines": {"$avg": "$MntWines"},
            "avg_meat": {"$avg": "$MntMeatProducts"},
            "avg_fruits": {"$avg": "$MntFruits"},
            "count": {"$sum": 1}
        }
    },
    {"$sort": {"avg_wines": -1}}
]

result = list(collection.aggregate(pipeline))

print(f"{'Education':<15} {'Count':>6} {'Avg Wines':>10} {'Avg Meat':>10} {'Avg Fruits':>10}")
print("-" * 55)
for r in result:
    print(f"{r['_id']:<15} {r['count']:>6} ${r['avg_wines']:>9.2f} ${r['avg_meat']:>9.2f} ${r['avg_fruits']:>9.2f}")

Education        Count  Avg Wines   Avg Meat Avg Fruits
-------------------------------------------------------
PhD                486 $   404.50 $   168.60 $    20.05
Master             370 $   333.08 $   163.38 $    21.65
Graduation        1127 $   284.27 $   179.49 $    30.77
2n Cycle           203 $   198.18 $   141.26 $    28.96
Basic               54 $     7.24 $    11.44 $    11.11


## Aggregation Pipeline: Campaign Acceptance Rate

In [17]:
pipeline = [
    {
        "$group": {
            "_id": None,
            "total": {"$sum": 1},
            "accepted_cmp1": {"$sum": "$AcceptedCmp1"},
            "accepted_cmp2": {"$sum": "$AcceptedCmp2"},
            "accepted_cmp3": {"$sum": "$AcceptedCmp3"},
            "accepted_cmp4": {"$sum": "$AcceptedCmp4"},
            "accepted_cmp5": {"$sum": "$AcceptedCmp5"},
            "final_response": {"$sum": "$Response"}
        }
    }
]

result = list(collection.aggregate(pipeline))[0]
total = result['total']

print(f"Total customers: {total}\n")
for cmp in ['cmp1','cmp2','cmp3','cmp4','cmp5']:
    key = f"accepted_{cmp}"
    pct = result[key] / total * 100
    print(f"Campaign {cmp[-1]} accepted: {result[key]:>4} ({pct:.1f}%)")

pct = result['final_response'] / total * 100
print(f"Final Response:    {result['final_response']:>4} ({pct:.1f}%)")

Total customers: 2240

Campaign 1 accepted:  144 (6.4%)
Campaign 2 accepted:   30 (1.3%)
Campaign 3 accepted:  163 (7.3%)
Campaign 4 accepted:  167 (7.5%)
Campaign 5 accepted:  163 (7.3%)
Final Response:     334 (14.9%)


## Load Full Collection into a Pandas DataFrame
Use `pd.DataFrame(list(data))` to convert MongoDB cursor to DataFrame

In [19]:
data = collection.find({})
df = pd.DataFrame(list(data))

# Drop MongoDB internal _id column for cleaner display
df = df.drop(columns=['_id'])

print("DataFrame shape:", df.shape)
df.head()

DataFrame shape: (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [21]:
# Quick statistical summary of numeric columns
df.describe().round(2)

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.00,2240.00,2216.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,...,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.00,2240.0,2240.0,2240.00
mean,5592.16,1968.81,52247.25,0.44,0.51,49.11,303.94,26.30,166.95,37.53,...,5.32,0.07,0.07,0.07,0.06,0.01,0.01,3.0,11.0,0.15
std,3246.66,11.98,25173.08,0.54,0.54,28.96,336.60,39.77,225.72,54.63,...,2.43,0.26,0.26,0.26,0.25,0.11,0.10,0.0,0.0,0.36
min,0.00,1893.00,1730.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3.0,11.0,0.00
25%,2828.25,1959.00,35303.00,0.00,0.00,24.00,23.75,1.00,16.00,3.00,...,3.00,0.00,0.00,0.00,0.00,0.00,0.00,3.0,11.0,0.00
50%,5458.50,1970.00,51381.50,0.00,0.00,49.00,173.50,8.00,67.00,12.00,...,6.00,0.00,0.00,0.00,0.00,0.00,0.00,3.0,11.0,0.00
75%,8427.75,1977.00,68522.00,1.00,1.00,74.00,504.25,33.00,232.00,50.00,...,7.00,0.00,0.00,0.00,0.00,0.00,0.00,3.0,11.0,0.00
max,11191.00,1996.00,666666.00,2.00,2.00,99.00,1493.00,199.00,1725.00,259.00,...,20.00,1.00,1.00,1.00,1.00,1.00,1.00,3.0,11.0,1.00


## DataFrame Analysis: Marital Status Distribution

In [22]:
marital_counts = df['Marital_Status'].value_counts()
print("Marital Status Breakdown:")
print(marital_counts.to_string())

Marital Status Breakdown:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2


## DataFrame Analysis: Top 5 Customers by Total Spending

In [23]:
spending_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
                 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']

df['TotalSpend'] = df[spending_cols].sum(axis=1)

top5 = df[['ID', 'Education', 'Income', 'TotalSpend']].sort_values('TotalSpend', ascending=False).head(5)
top5['Income'] = top5['Income'].apply(lambda x: f"${x:,.0f}" if pd.notna(x) else "N/A")
top5['TotalSpend'] = top5['TotalSpend'].apply(lambda x: f"${x:,.0f}")
print(top5.to_string(index=False))

  ID  Education  Income TotalSpend
5350     Master $90,638     $2,525
5735     Master $90,638     $2,525
1763 Graduation $87,679     $2,524
4580 Graduation $75,759     $2,486
4475        PhD $69,098     $2,440


---
## Conclusion

This notebook demonstrated key MongoDB operations on the `marketing_campaign` collection:

- **`count_documents({})`** — count all or filtered records
- **`find(query, projection).limit(n)`** — retrieve filtered fields
- **`$gt`, `$lt`, `$gte`, `$lte`** — comparison operators in queries
- **`aggregate(pipeline)`** — for grouping, max/min, averages
- **`pd.DataFrame(list(collection.find({})))`** — load into pandas for analysis

**Key findings:**
- Use `$gt` / `$lt` to filter by numeric thresholds (income, spending, recency)
- Combine multiple conditions in one query dict for AND logic
- Aggregation pipelines are required for grouped statistics
